<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_02_pinn_soft_and_hard.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 · Notebook 02 — The Walls, Soft and Hard

**Paired with L7.1 · Fundamentals of PINNs**

A physics-informed network learns the slot's temperature for **every current
from 10 to 45 A at once** — the current is one of its inputs — and is scored
against notebook 01's ground truth. It is trained twice, with the wall
condition imposed the two ways L7.1 teaches:

* **soft** — a penalty in the loss, asked for at points on the walls (L7.1's
  **Soft Enforcement** slide);
* **hard** — built into the output with a mask that is zero on the walls
  (L7.1's **Hard Enforcement** slide).

Then you shrink the network until it stops being accurate enough, and put the
winner against finite differences on accuracy and on time.

## What you will do

1. Write the residual: the slot's equation with the network in place of $\theta$.
2. Train the same network with soft walls and with hard walls.
3. Score both against notebook 01's ground truth, in kelvin.
4. Find the smallest network that is still accurate enough.
5. Compare with finite differences, and say who wins.

## One thing to watch

The network takes current as an input, so one training answers every current.
That is the only thing it has to offer against finite differences, which solve
the slot in milliseconds. Section 5 measures whether it is enough.

---

## 0 · Setup

**What these cells do.** Fetch the library files, keep results in your Google Drive, import the modules, and load notebook 01's ground truth.

**How.** As in notebook 01, then `cc.needed` makes sure `nb01_truth.npz` is there — on Colab without Drive it asks you to upload it.

In [ ]:
# files-cell v2 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository on every run, so a file left
# over from an earlier session cannot shadow it. Run this cell first.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    # on Colab always re-fetch: /content outlives a session and a stale copy
    # would silently shadow an updated one. Locally the repo's own files stay.
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
# a module imported before this cell would keep the previous file's contents
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex07.1_outputs")

In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)  # fix the random start so a run repeats
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

cc.needed("nb01_truth.npz")                              # on Colab without Drive, asks for it
truth = np.load(os.path.join(cc.OUTPUT_DIR, "nb01_truth.npz"))
print("ground truth at", ", ".join(f"{I:.1f}" for I in truth["currents"]), "A")

---

## 1 · The network, and its two walls

**What this cell does.** Defines the network and draws the points it trains on.

**How.** The network sees three numbers: $\xi = x/a$ and $\eta = y/b$, which run from −1 to 1 across the slot, and $s$, the current scaled to the same range. Its output is scaled to the size of the answer,

$$\hat\theta = 20\ \text{K}\cdot\left(\frac{I}{I_{\text{rated}}}\right)^2 N(\xi, \eta, s),$$

because the rise grows roughly as $I^2$: the network then only has to learn the shape and the few per cent on top. With **hard** walls the output is also multiplied by the mask $(1 - \xi^2)(1 - \eta^2)$, zero on every wall. The points: 2000 inside the slot and the current range, by Latin hypercube, and 400 on the walls for the soft penalty. Both sets stay fixed, because L-BFGS needs the same points at every step.

In [ ]:
a, b = pb.A_HALF, pb.B_HALF
I_LO, I_HI = pb.I_RANGE
S = 20.0                                                  # K: the size of the answer
Q = pb.heat_coefficients(pb.I_RATED)[0]                   # W/m^3: the size of the heat

def current(s):
    return I_LO + (s + 1) / 2 * (I_HI - I_LO)             # s in -1..1 back to amperes

class Slot(torch.nn.Module):                              # theta = S (I/I_rated)^2 N, masked if hard
    def __init__(self, hard, width=32, layers=4):
        super().__init__()
        self.hard = hard
        self.net = MLP(n_in=3, n_hidden=width, n_layers=layers)   # tanh network: xi, eta, s in

    def forward(self, p):
        xi, eta, s = p[:, :1], p[:, 1:2], p[:, 2:]
        out = S * (current(s) / pb.I_RATED) ** 2 * self.net(p)
        return out * (1 - xi ** 2) * (1 - eta ** 2) if self.hard else out   # the mask

box = ((-1, 1), (-1, 1), (-1, 1))                         # xi, eta and the current
inner = to_tensor(interior_points(2000, box, method="lhs", seed=1), requires_grad=True)  # fixed
w2 = boundary_points(100, ((-1, 1), (-1, 1)), seed=1)     # 400 points on the four walls
walls = to_tensor(np.hstack([w2, np.random.default_rng(1).uniform(-1, 1, (len(w2), 1))]))  # each wall point at a random current
print("inside:", tuple(inner.shape), " on the walls:", tuple(walls.shape))

---

## 2 · The residual

**What this cell does.** Defines the residual: the slot's equation, $k_{\text{eff}}\nabla^2\theta + q(\theta, I)$, with the network in place of $\theta$, divided by the rated heat so that it is a number of order one.

**How.** `d2(th, p, 0)` is the second derivative in $\xi$. Since $\xi = x/a$, a derivative in $x$ is a derivative in $\xi$ divided by $a$ — twice over for a second derivative, so $\partial^2/\partial x^2 = (1/a^2)\,\partial^2/\partial\xi^2$. The heat is `pb.heat_source`, the function notebook 01 checked.

In [ ]:
# TODO: write the residual, scaled by the rated heat Q.
#
#   def residual(model, p):
#       th = model(p)                                                     # K
#       lap = d2(th, p, 0) / a ** 2 + d2(th, p, 1) / b ** 2               # in metres
#       return (pb.K_EFF * lap + pb.heat_source(th, current(p[:, 2:]))) / Q
#
# Leave out the 1/a^2 and 1/b^2 and the network solves a slot 2 m by 2 m.

raise NotImplementedError("Write residual(model, p)")

---

## 3 · Train, soft and hard

**What this cell does.** Trains the same network twice, on the same points with the same budget: once with soft walls, once with hard.

**How.** The soft loss is the residual squared plus the wall rise squared, $\mathcal{L} = \overline{r^2} + w\,\overline{(\hat\theta_{\text{wall}}/S)^2}$ with $w = 1$; the hard loss is the residual alone, because the mask has already met the walls. Both use the course's schedule, Adam then L-BFGS. About a minute each on a CPU.

In [ ]:
nets, train_time = {}, {}
for name, hard in (("soft", False), ("hard", True)):
    set_seed(88)                                           # the same start for both
    model = Slot(hard).to(DEVICE)                          # four layers of 32

    def loss():                                            # what train_two_stage minimises
        l = residual(model, inner).pow(2).mean()           # the equation, inside
        if not hard:
            l = l + (model(walls) / S).pow(2).mean()       # the soft wall: asked, w = 1
        return l

    t0 = time.perf_counter()
    train_two_stage(model, loss, adam_steps=1500, lbfgs_steps=150, lr=2e-3, report_every=0)  # Adam, then L-BFGS
    nets[name], train_time[name] = model, time.perf_counter() - t0
    print(f"{name}: trained in {train_time[name]:.0f} s")

---

## 4 · Score against the ground truth

**What this cell does.** Evaluates both networks at notebook 01's six currents, on the same grid, and prints the worst error in kelvin — inside the slot and on its walls. Then it maps the error at 45 A.

**How.** The ground truth is on a 161 × 321 grid. For each current the cell builds the network's inputs on that grid, runs it, and subtracts. The maps show the error with its sign — blue where the network is too cold, red where it is too hot, white where it is right.

In [ ]:
X, Y, P = grid_points(161, 321, pb.DOMAIN)                # notebook 01's grid
def evaluate(model, I, P=P):
    s = 2 * (I - I_LO) / (I_HI - I_LO) - 1
    p = to_tensor(np.hstack([P / [a, b], np.full((len(P), 1), s)]))  # scaled point and current, the network's inputs
    with torch.no_grad():                                  # evaluate only, no graph
        return to_numpy(model(p)).ravel()

worst, wall = {}, {}
for name, model in nets.items():
    errs = [np.abs(evaluate(model, I) - f.ravel()).max() for I, f in zip(truth["currents"], truth["fields"])]
    worst[name] = max(errs)
    with torch.no_grad():                                  # evaluate only, no graph
        wall[name] = model(walls).abs().max().item()
    print(f"{name}: worst error {worst[name]:.3f} K over the six currents, "
          f"worst on the walls {wall[name]:.3f} K")

fig, axes = plt.subplots(1, 2, figsize=(8.6, 5.6))
for ax, name in zip(axes, nets):
    pb.plot_field(evaluate(nets[name], 45.0) - truth["fields"][-1].ravel(), 161, 321, ax=ax,
                  title=f"{name}: network minus truth, 45 A", label="error  [K]", cmap="coolwarm")
plt.tight_layout(); plt.show()

**What you should see.** The hard network within about **0.02 K** everywhere,
at every current, and exactly zero on the walls. The soft one about **0.2 K**
off on the walls — and, worse, more than a kelvin wrong inside.

That second number is the lesson. A wall that is a little too warm does not
stay a boundary problem: the temperature inside is fixed by the walls, so the
error leaks into the whole slot. The soft network traded the wall against the
equation, as a penalty lets it, and paid for it everywhere. The mask never
offers that trade.

---

## 5 · The smallest network that will do

**What this cell does.** Trains the hard network at two smaller sizes and prints how accurate each is, how many parameters it has, and how long it took.

**How.** The same points, schedule and seed; only the width and depth change. A smaller network evaluates faster, which is the one number a trained network can win on.

In [ ]:
sizes = {"32 x 4": nets["hard"]}
for width, layers in ((16, 3), (8, 3)):
    set_seed(88)                                           # the same start as the big one
    model = Slot(True, width, layers).to(DEVICE)           # a smaller hard network

    def loss():
        return residual(model, inner).pow(2).mean()        # hard walls: the equation alone

    t0 = time.perf_counter()
    train_two_stage(model, loss, adam_steps=1500, lbfgs_steps=150, lr=2e-3, report_every=0)  # Adam, then L-BFGS
    sizes[f"{width} x {layers}"] = model
    train_time[f"{width} x {layers}"] = time.perf_counter() - t0
train_time["32 x 4"] = train_time["hard"]

sweep = {}
for name, model in sizes.items():
    err = max(np.abs(evaluate(model, I) - f.ravel()).max() for I, f in zip(truth["currents"], truth["fields"]))
    sweep[name] = (parameter_count(model), err, train_time[name])  # size, worst error, training time
    print(f"  {name:7s} {sweep[name][0]:6d} parameters   worst error {err:.3f} K   trained in {train_time[name]:3.0f} s")

**What you should see.** Sixteen by three — a fifth of the parameters — still
within a few hundredths of a kelvin; eight by three about two tenths out. So there
is a smallest network, and below it the accuracy goes quickly.

---

## 6 · Who wins

**What this cell does.** Puts the hard network against finite differences at the same accuracy, on the same nodes, and prints the time per current, the training time, and how many currents it takes before the training has paid for itself.

**How.** It scores finite differences exactly as it scored the network — the worst error over the same six currents — and picks the coarsest grid at least as accurate. Then it times one solve on that grid — grid, matrix and solve, the middle of five runs — and the network evaluating the same nodes. The break-even is the training time divided by the time saved per current.

In [ ]:
e_net = worst["hard"]
fdm_err = {}                                                        # FDM scored as the network was:
for g in (11, 21, 41, 81):                                          # worst error over the six currents
    s = 160 // (g - 1)                                              # the truth's nodes on this grid
    fdm_err[g] = max(np.abs(pb.fdm_solve(I, g)[2] - f[::s, ::s]).max()
                     for I, f in zip(truth["currents"], truth["fields"]))
nx = next(g for g, e in fdm_err.items() if e <= e_net)             # coarsest grid as accurate

def middle(fn, reps=5):
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

t_fdm = middle(lambda: pb.fdm_solve(28.0, nx))                      # grid + matrix + solve
Xg, Yg, Pg = grid_points(nx, 2 * nx - 1, pb.DOMAIN)                # the same nodes
t_net = middle(lambda: evaluate(nets["hard"], 28.0, Pg))            # a trained network
t_train = train_time["hard"]
break_even = t_train / max(t_fdm - t_net, 1e-9)

print(f"{'':30s}{'worst error':>12s}{'per current':>14s}{'training':>11s}")
print(f"  {'finite differences, ' + f'{nx} x {2*nx-1}':28s}{fdm_err[nx]:11.3f} K"
      f"{t_fdm*1e3:11.2f} ms{'-':>11s}")
print(f"  {'PINN, hard walls':28s}{e_net:11.3f} K{t_net*1e3:11.2f} ms{t_train:9.0f} s")
print(f"  {'PINN, soft walls':28s}{worst['soft']:11.3f} K{'':>14s}{train_time['soft']:9.0f} s")
print(f"\nthe trained network is {t_fdm / t_net:.0f} x faster per current;")
print(f"its training pays for itself after about {break_even:,.0f} currents")

**What you should see.** A finite-difference grid as accurate as the hard
network, solved in a few milliseconds. The trained network several times faster
per current — and a training time of a minute or so, which it only earns back
after **about ten thousand** currents.

So on this slot **finite differences win**: one solve is almost free, and the
feedback of hot copper does not change that, because each current is still one
linear solve. The network wins only where many questions are asked of one
training — a design sweep, a controller asking every few milliseconds, a
digital twin — or where a grid is costly: awkward 3-D shapes, unknown
parameters found from data, many inputs at once. That is where the rest of
Part 2 goes.

---

## 7 · Save

**What this cell does.** Saves the scores, the sweep and the timings for the report.

In [ ]:
path = cc.output_path("nb02_pinn.npz")
np.savez(path, worst_soft=worst["soft"], worst_hard=worst["hard"],
         wall_soft=wall["soft"], train_soft=train_time["soft"], train_hard=train_time["hard"],
         sweep_names=np.array(list(sweep)), sweep=np.array(list(sweep.values())),
         fdm_grid=nx, t_fdm=t_fdm, t_net=t_net, break_even=break_even)
print("wrote", path)
cc.saved(path)

---

## 8 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which.

1. The soft network was about 0.2 K off on the walls and more than a kelvin off inside. Explain from the loss why a soft wall is met only nearly, and why its error does not stay on the wall.
   *→ L7.1 Q7*
2. What does the mask guarantee, from the first step of training? What would it cost for a slot with a rounded bottom, or a wall held at 80 °C instead of the iron's temperature?
   *→ L7.1 Q9*
3. The network trained on 2000 points inside the slot. How does that compare with $N^*$ for four layers of 32 with three inputs, and what happened to the accuracy as the network shrank?
   *→ L7.1 Q2*
4. Who won, and by how much? For which job would you train a network instead of solving with finite differences, and how many currents would it take before the training paid?
   *→ L7.1 Q4*

*Write your answers here. You will copy them into the report in notebook 04, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 03](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_03_helmholtz.ipynb)**, which explains why the hot spot grows faster than $I^2$: hot copper turns the slot's Poisson equation into a Helmholtz equation, and a network finds its eigenvalue.

<!-- two-forms -->
**Open notebook 03 as**

- **[the exercise](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_03_helmholtz.ipynb)**, where you write the missing lines, or
- **[the light version](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_03_helmholtz_light.ipynb)**, with every cell written out.